In [2]:
# always gotta start with the
import numpy as np

In [3]:
#--------------------------------------------------------
# Section 2.1 - Preparing the Calculations
#--------------------------------------------------------

# first, express the gravitiational parameter of the sun in canonical units, so it is equal to 1
mu = 1.0

# next, we will need to rotate the orbit of body A so that it has zero inclination, argument of periapsis, and longitude of the ascending node
# we will use Euler Angle matrices to perform this rotation, in the standard form R = R3(-w1) R1(-i1) R3(-Om1)

def R1(phi):
    #writing np.cos and np.sin so much is very tiring so this is a shortcut
    c, s = np.cos(phi), np.sin(phi)
    return np.array([[1,0,0],
                     [0,c,-s],
                     [0,s,c]])
def R3(phi):
    c, s = np.cos(phi), np.sin(phi)
    return np.array([[c,-s,0],
                     [s,c,0],
                     [0,0,1]])
    
# we also want functions that can convert our orbital elements (a, e, i, omega, Omega) into states (r and v vectors) and vice versa
def oe_to_state(a, e, i, omega, Omega, nu):
    # use semi-latus rectum to account for hyperbolic orbits
    p = a * (1-e**2)
    r = p / (1 + e * np.cos(nu))
    
    r_pf = np.array([r*np.cos(nu),
                     r*np.sin(nu),
                     0])
    
    # Velocity in perifocal frame using p
    v_pf = np.array([
        -np.sqrt(mu / p) * np.sin(nu),
         np.sqrt(mu / p) * (e + np.cos(nu)),
         0
    ])
    
    Q = R3(Omega) @ R1(i) @ R3(omega)
    r_vec = Q @ r_pf
    v_vec = Q @ v_pf
    
    return r_vec, v_vec

def state_to_oe(r, v):
    h = np.cross(r, v)
    h_norm = np.linalg.norm(h)
    
    i = np.arccos(h[2]/h_norm)
    
    k = np.array([0,0,1])
    n = np.cross(k, h)
    n_norm = np.linalg.norm(n)
    
    Omega = np.arctan2(n[1], n[0])
    
    e_vec = ((np.linalg.norm(v)**2 - mu/np.linalg.norm(r))*r - np.dot(r,v)*v)/mu
    e = np.linalg.norm(e_vec)
    
    omega = np.arctan2(np.dot(h, np.cross(n, e_vec))/(h_norm*n_norm), np.dot(n, e_vec)/n_norm)
    
    a = 1/(2/np.linalg.norm(r) - np.linalg.norm(v)**2/mu)
    
    return a, e, i, omega, Omega

# now that we have these, we can rotate our reference frame such that A has no inclination, AOP, and LAN, 
# and express both A and B's elements in this new form
def transform_frame(oeA, oeB):
    a1, e1, i1, omega1, Omega1 = oeA
    a2, e2, i2, omega2, Omega2 = oeB

    # transforms from A frame to new frame
    Q_A = R3(Omega1) @ R1(i1) @ R3(omega1)

    # Rotate B frame to match
    R = Q_A.T

    # Use nu = 0 (any consistent value works)
    rB, vB = oe_to_state(a2, e2, i2, omega2, Omega2, 0.0)

    rB_rot = R @ rB
    vB_rot = R @ vB

    return state_to_oe(rB_rot, vB_rot)

In [4]:
#--------------------------------------------------------
# Section 2.2 - Scanning the Orbits
#--------------------------------------------------------

# function that replicates eq. 1 in a more generalized form
def radius(a, e, nu):
    return a*(1 - e**2) / (1 + e*np.cos(nu))

# find the Cartesian coordinates of bodies A and B, based on their respective true anomalies L and nu (useful for section 2.3)
def coords_A(aA, eA, L):
    rA = radius(aA, eA, L)
    return np.array([rA*np.cos(L),
                     rA*np.sin(L),
                     0.0])

def coords_B(aB, eB, iB, omegaB, OmegaB, nu):
    rB = radius(aB, eB, nu)

    cO, sO = np.cos(OmegaB), np.sin(OmegaB)
    cw, sw = np.cos(omegaB+nu), np.sin(omegaB+nu)
    ci, si = np.cos(iB), np.sin(iB)

    x = rB*(cO*cw - sO*sw*ci)
    y = rB*(sO*cw + cO*sw*ci)
    z = rB*(sw*si)

    return np.array([x,y,z])

# find the meridional distance between A and B, as defined by equations 2-6
def meridional_distance(aA,eA,aB,eB,iB,omegaB,OmegaB,nu):

    rB_vec = coords_B(aB,eB,iB,omegaB,OmegaB,nu)
    xB,yB,zB = rB_vec

    rhoB = np.sqrt(xB**2 + yB**2)

    if rhoB == 0:
        return np.inf

    cosL = xB / rhoB
    sinL = yB / rhoB
    L = np.arctan2(sinL, cosL)

    rA = radius(aA,eA,L)

    D = np.sqrt(zB**2 + (rhoB - rA)**2)
    return D

# using the meridional distance function, scan through values of nu to find the local minima of the meridional distance
# the paper said 0.12 radians but I have a good PC so I will make the step smaller
def scan_meridional(aA,eA,aB,eB,iB,omegaB,OmegaB,step=0.01):
    # a list of nu values from 0 to 2pi with the given step size
    nus = np.arange(0, 2*np.pi, step)
    Ds = []
    for nu in nus:
        Ds.append(meridional_distance(aA, eA, aB, eB, iB, omegaB, OmegaB, nu))
    
    Ds = np.array(Ds)
    minima = []
    n = len(Ds)
    
    # Fixed loop to handle circular wrapping (0 and 2pi)
    for k in range(n):
        left = Ds[k-1] # Python handles index -1 correctly (wraps to end)
        right = Ds[(k+1) % n]
        if Ds[k] < left and Ds[k] < right:
            minima.append(nus[k])
            
    return minima

# the above is only a meridional distance, the below function will find the true 3D distance between A and B at some true anomalies L and nu
# as per equations 7-10
def full_distance(aA,eA,aB,eB,iB,omegaB,OmegaB,L,nu):
    rA = coords_A(aA,eA,L)
    rB = coords_B(aB,eB,iB,omegaB,OmegaB,nu)
    return np.linalg.norm(rA - rB)

In [5]:
#--------------------------------------------------------
# Section 2.3 - Parallel Tuning
#--------------------------------------------------------

# the paper describes two phases of tuning, an initial one and a final one, with the difference being step sizes
# this seemed unnecessary to me, so we will simply run the initial tuning with a smaller target step, until the desired step size is reached
def parallel_tuning_initial(aA, eA, aB, eB, iB, omegaB, OmegaB, L0, nu0, step_init=0.01, target_step=1e-10, reduction_factor=0.5):
    
    L = L0
    nu = nu0
    step = step_init
    # tuning repeats as long as the step is larger than our target
    while step > target_step:

        improved = True

        while improved:
            improved = False

            best_L = L
            best_nu = nu
            best_D2 = full_distance(aA, eA, aB, eB, iB, omegaB, OmegaB,L, nu)**2

            # Evaluate all 9 points, which represent all distance combinations of L, L+/- step, nu, and nu +/- step
            for dL in (-step, 0.0, step):
                for dnu in (-step, 0.0, step):

                    L_test = (L + dL) % (2*np.pi)
                    nu_test = (nu + dnu) % (2*np.pi)
                    # for each combination of places A and B could be, find the distance between them
                    D2 = full_distance(aA, eA, aB, eB, iB, omegaB, OmegaB, L_test, nu_test)**2

                    # if the distance (squared) we find is smaller than the previous best, set it as the new best
                    if D2 < best_D2:
                        best_D2 = D2
                        best_L = L_test
                        best_nu = nu_test
                        improved = True
            # set the L and nu at which the new best was found as our new reference point
            L, nu = best_L, best_nu

        # Reduce step size and repeat until target step size is reached
        step *= reduction_factor

    return np.sqrt(best_D2), L, nu

In [6]:
# ------------------------------------------------------------
# Section 2.4 - Finding the MOID
# ------------------------------------------------------------

def compute_moid(aA,eA,aB,eB,iB,omegaB,OmegaB):
    # find the minima of the meridional distance
    minima = scan_meridional(aA,eA,aB,eB,iB,omegaB,OmegaB)

    # there is a small case that there is no minimum
    if len(minima) == 0:
        return None
    # placeholder value for moid
    moid = np.inf
    L_moid = 0
    nu_moid = 0
    # for each minimum meridional distance, perform parallel tuning to find the minimum distance
    for nu0 in minima:

        # corresponding L from meridional geometry
        rB_vec = coords_B(aB,eB,iB,omegaB,OmegaB,nu0)
        xB,yB,_ = rB_vec
        L0 = np.arctan2(yB,xB)

        # we only care aboout the distance value from parallel tuning
        D,L,nu = parallel_tuning_initial(aA,eA,aB,eB,iB,omegaB,OmegaB,L0,nu0)

        # repeat for all local minima to find the true moid, and the accompanying true anomalies
        if D < moid:
            moid = D
            L_moid = L
            nu_moid = nu

    return moid, L_moid, nu_moid

In [7]:
# ------------------------------------------------------------
# Testing
# ------------------------------------------------------------

# body A is asteroid (21) Lutetia (denoted with L), body B is test no. 1 on page 10 (denoted with T)
qL = 2.036 #AU
eL = 0.164 #dimensionless
iL = 0
wL = 0
WL = 0

qT = 2.55343183 # AU
eT = 0.0777898  # dimensionless
iT = 10.58785   # degrees
wT = 80.35052
WT = 72.14554

# convert q (perihelion distance) to semimajor axis using the following
def QtoA(q,e):
    return q/(1-e)

oea = [QtoA(qL,eL),eL,np.radians(iL),np.radians(wL),np.radians(WL)]
oeb = [QtoA(qT,eT),eT,np.radians(iT),np.radians(wT),np.radians(WT)]

aA, eA, iA, omegaA, OmegaA = [np.float64(QtoA(qL,eL)),np.float64(eL),np.float64(0),np.float64(0),np.float64(0)]
aB, eB, iB, omegaB, OmegaB = transform_frame(oea, oeb)

# testing to see that only omegaB and OmegaB have changed, and that aB=A2, eB=e2, and iB=i2 (they do)
#print (oeb)
#print([aB, eB, iB, omegaB, OmegaB])

compute_moid(aA,eA,aB,eB,iB,omegaB,OmegaB)
# this gives 0.181 AU, but the correct answer according to the paper is closer to 0.134 AU

(np.float64(0.18111583481048013),
 np.float64(4.1086548544214425),
 np.float64(1.448380115032196))

In [8]:
# combine all the testing code above into one concise function with better formatting
# this takes in the angular quanitites as degrees
def moid_nice(a1,e1,i1,w1,W1,a2,e2,i2,w2,W2):
    # orbital elements defined for body A and B
    oea = [a1,e1,np.radians(i1),np.radians(w1),np.radians(W1)]
    oeb = [a2,e2,np.radians(i2),np.radians(w2),np.radians(W2)]
    
    # convert to new orbital elements in new reference frame
    aA, eA, iA, omegaA, OmegaA = [np.float64(a1),np.float64(e1),np.float64(0),np.float64(0),np.float64(0)]
    aB, eB, iB, omegaB, OmegaB = transform_frame(oea, oeb)

    moid, L, nu = compute_moid(aA,eA,aB,eB,iB,omegaB,OmegaB)
    print("The MOID between these two bodies is " + str(round(moid,3)) + " AU, or " + format(round(moid * 149597870.691), ",") + " kilometers.")
    print("The MOID occurred at a true anomaly of A of " + str(round(L*180/np.pi,3)) + " degrees, and a true anomaly of B of " + str(round(nu*180/np.pi,3)) + " degrees.")
    #return moid


In [9]:
# question 1
moid_nice(1.4765067E+08/149597870.7, 9.1669995E-03, 4.2422693E-03, 6.64375167E+01, 1.4760836E+01,
          1.3793939E+08/149597870.7, 1.9097084E-01, 3.3356539E+00, 1.2919949E+02 , 2.0381969E+02)

The MOID between these two bodies is 0.006 AU, or 847,897 kilometers.
The MOID occurred at a true anomaly of A of 127.967 degrees, and a true anomaly of B of 236.155 degrees.


In [10]:
# question 2
moid_nice(1.4765067E+08/149597870.7, 9.1669995E-03, 4.2422693E-03, 6.64375167E+01, 1.4760836E+01,
          3.7680703E+08/149597870.7, 6.6164147E-01, 3.4001497E+00, 1.3429905E+02 , 2.7147904E+02)

The MOID between these two bodies is 0.002 AU, or 264,453 kilometers.
The MOID occurred at a true anomaly of A of 11.927 degrees, and a true anomaly of B of 47.35 degrees.


In [11]:
# question 3
moid_nice(1.3793939E+08/149597870.7, 1.9097084E-01, 3.3356539E+00, 1.2919949E+02 , 2.0381969E+02,
          3.7680703E+08/149597870.7, 6.6164147E-01, 3.4001497E+00, 1.3429905E+02 , 2.7147904E+02)

The MOID between these two bodies is 0.05 AU, or 7,530,658 kilometers.
The MOID occurred at a true anomaly of A of 124.297 degrees, and a true anomaly of B of 51.506 degrees.


In [124]:
# question 4
moid_nice(1.4765067E+08/149597870.7, 9.1669995E-03, 4.2422693E-03, 6.64375167E+01, 1.4760836E+01,
          -3.9552667E+07/149597870.7, 6.1469268E+00, 1.7512507E+02, 1.2817255E+02, 3.2228906E+02)

The MOID between these two bodies is 0.378 AU, or 56,610,143 kilometers.
The MOID occurred at a true anomaly of A of 112.73 degrees, and a true anomaly of B of 0.219 degrees.


In [ ]:
# it is worth remembering that the MOID is NOT the closest that these two bodies ever get to each other
# it is the closest their orbital paths get to each other
# we talked about this in the lecture